[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/02_Serialization/Serialization_Deep_Dive.ipynb)

# 1.2 ONNX Serialization — Deep Dive

ONNX is built on **Protocol Buffers (protobuf)**, Google's language-neutral serialization format. Every ONNX object — models, graphs, nodes, tensors — can be serialized to bytes and deserialized back, enabling portable model distribution across languages, frameworks, and platforms.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Protocol Buffers Foundation](#section-1) | What protobuf is and why ONNX uses it |
| 2 | [The Serialization Pipeline](#section-2) | Object ↔ bytes ↔ file round-trip |
| 3 | [Model Serialization](#section-3) | Saving and loading `ModelProto` |
| 4 | [Tensor Serialization](#section-4) | Saving and loading `TensorProto` with `numpy_helper` |
| 5 | [Anatomy of a .onnx File](#section-5) | Byte-level structure visualization |
| 6 | [All Serializable Proto Types](#section-6) | Complete catalog with examples |
| 7 | [File Size Analysis](#section-7) | What drives model file size |
| 8 | [The 2 GB Protobuf Limit](#section-8) | External data format for large models |
| 9 | [Key Takeaways](#section-9) | Summary and best practices |

In [ ]:
# Uncomment for Colab:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import os
import sys

from onnx import TensorProto, load, save
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info)
from onnx.checker import check_model
from onnx import numpy_helper

print('Setup complete!')

<a id='section-1'></a>
## Section 1: Protocol Buffers Foundation

### What Is Protobuf?

Protocol Buffers (protobuf) is a binary serialization format developed by Google. It defines data structures in `.proto` schema files, then generates code in any language (Python, C++, Java, etc.) to read and write those structures efficiently.

ONNX chose protobuf for several critical reasons:

| Property | Benefit for ONNX |
|----------|------------------|
| **Language-neutral** | Same `.onnx` file works with Python, C++, Java, C#, Rust, etc. |
| **Compact binary format** | Models are stored efficiently (much smaller than JSON/XML) |
| **Schema-enforced** | The `.proto` file defines the exact structure of every ONNX object |
| **Backward compatible** | New fields can be added without breaking old readers |
| **Fast parsing** | Deserialization is orders of magnitude faster than text formats |

### The ONNX Proto Schema

The core ONNX schema lives in a file called `onnx.proto3`. It defines the hierarchy:

```
onnx.proto3
├── ModelProto           ← Top-level container
│   ├── ir_version       : int64
│   ├── opset_import     : repeated OpsetIdProto
│   ├── producer_name    : string
│   ├── graph            : GraphProto
│   │   ├── node         : repeated NodeProto
│   │   ├── input        : repeated ValueInfoProto
│   │   ├── output       : repeated ValueInfoProto
│   │   └── initializer  : repeated TensorProto
│   └── functions        : repeated FunctionProto
├── TensorProto          ← Tensor data + metadata
│   ├── dims             : repeated int64
│   ├── data_type        : int32
│   └── raw_data         : bytes
└── ValueInfoProto       ← Type information (no data)
    ├── name             : string
    └── type             : TypeProto
```

Every Python ONNX object you create (`ModelProto`, `GraphProto`, `NodeProto`, etc.) is a protobuf message generated from this schema. This is why they all share the same two serialization methods: `SerializeToString()` and `ParseFromString()`.

<a id='section-2'></a>
## Section 2: The Serialization Pipeline

The complete round-trip has three stages. Understanding this pipeline is essential because it's how models move between training frameworks, optimization tools, and inference engines.

```
┌──────────────────┐     SerializeToString()     ┌──────────┐     write()      ┌────────────┐
│  Python Objects   │ ──────────────────────────► │  bytes   │ ──────────────► │  .onnx file │
│  (ModelProto,     │                             │  (in     │                 │  (on disk)  │
│   TensorProto,    │ ◄────────────────────────── │  memory) │ ◄────────────── │             │
│   etc.)           │     ParseFromString()       │          │     read()      │             │
└──────────────────┘                              └──────────┘                 └────────────┘
```

### The Two Fundamental Methods

Every protobuf message in ONNX supports exactly two serialization methods:

| Method | Direction | Input → Output |
|--------|-----------|----------------|
| `SerializeToString()` | Object → bytes | Converts the in-memory protobuf message to a compact binary string |
| `ParseFromString(data)` | bytes → Object | Populates a protobuf message from binary data |

These are protobuf-native methods, not ONNX-specific. The ONNX library adds convenience wrappers like `onnx.save()` and `onnx.load()` that combine serialization with file I/O, but underneath they call the same protobuf methods.

### Convenience Functions

ONNX provides higher-level functions that handle the full pipeline:

| Function | Equivalent To |
|----------|---------------|
| `onnx.save(model, path)` | `open(path, 'wb').write(model.SerializeToString())` |
| `onnx.load(path)` | `ModelProto(); m.ParseFromString(open(path, 'rb').read())` |
| `onnx.load_model_from_string(data)` | `ModelProto(); m.ParseFromString(data)` |

<a id='section-3'></a>
## Section 3: Model Serialization

Let's build a model, serialize it, save to disk, load it back, and verify the round-trip is lossless.

In [ ]:
# Build a simple linear regression model
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

graph = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['Y'])],
    'lr', [X, A, B], [Y])
onnx_model = make_model(graph)
check_model(onnx_model)

print(f'Model created: {len(onnx_model.graph.node)} nodes, '
      f'{len(onnx_model.graph.input)} inputs')

In [ ]:
# ── Method 1: Low-level protobuf serialization ──────────────────────
serialized_bytes = onnx_model.SerializeToString()

print(f'Serialized size: {len(serialized_bytes)} bytes')
print(f'First 50 bytes (hex): {serialized_bytes[:50].hex()}')
print(f'Type: {type(serialized_bytes).__name__}')

# Save to disk
with open('model_v1.onnx', 'wb') as f:
    f.write(serialized_bytes)
print(f'\nSaved to model_v1.onnx ({os.path.getsize("model_v1.onnx")} bytes on disk)')

In [ ]:
# ── Method 2: High-level onnx.save() / onnx.load() ──────────────────
save(onnx_model, 'model_v2.onnx')
loaded_model = load('model_v2.onnx')

# Verify round-trip: the serialized bytes must be identical
original_bytes = onnx_model.SerializeToString()
loaded_bytes = loaded_model.SerializeToString()

print(f'Original size:  {len(original_bytes)} bytes')
print(f'Loaded size:    {len(loaded_bytes)} bytes')
print(f'Byte-identical: {original_bytes == loaded_bytes}')
print(f'Graph name:     {loaded_model.graph.name}')
print(f'Nodes:          {[n.op_type for n in loaded_model.graph.node]}')

### Verifying Functional Equivalence

Beyond byte equality, let's confirm the loaded model produces identical inference results.

In [ ]:
import onnxruntime as ort

sess_orig = ort.InferenceSession(
    onnx_model.SerializeToString(), providers=['CPUExecutionProvider'])
sess_loaded = ort.InferenceSession(
    loaded_model.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.random.randn(5, 3).astype(np.float32)
a = np.random.randn(3, 2).astype(np.float32)
b = np.random.randn(1, 2).astype(np.float32)
feeds = {'X': x, 'A': a, 'B': b}

res_orig = sess_orig.run(None, feeds)[0]
res_loaded = sess_loaded.run(None, feeds)[0]

print(f'Original  result: {res_orig[0]}')
print(f'Loaded    result: {res_loaded[0]}')
print(f'Max difference:   {np.abs(res_orig - res_loaded).max()}')
print(f'Functionally identical: {np.allclose(res_orig, res_loaded)}')

<a id='section-4'></a>
## Section 4: Tensor Serialization

Individual tensors can be serialized independently of a model. This is useful for:
- **Test data**: saving reference inputs/outputs for validation
- **Weights**: distributing pre-trained parameters separately
- **Calibration data**: storing representative samples for quantization

### The `numpy_helper` Bridge

ONNX provides `numpy_helper` to convert between NumPy arrays and ONNX `TensorProto` objects:

```
numpy.ndarray  ──► numpy_helper.from_array() ──► TensorProto ──► SerializeToString() ──► bytes
bytes ──► ParseFromString() ──► TensorProto ──► numpy_helper.to_array() ──► numpy.ndarray
```

In [ ]:
from onnx.numpy_helper import from_array, to_array

# Create a tensor with known values
original_array = np.array([[1.0, 2.0, 3.0],
                           [4.0, 5.0, 6.0]], dtype=np.float32)

# NumPy → TensorProto
tensor_proto = from_array(original_array, name='my_weights')

print('TensorProto fields:')
print(f'  name:      {tensor_proto.name}')
print(f'  dims:      {list(tensor_proto.dims)}')
print(f'  data_type: {tensor_proto.data_type} (1=FLOAT32)')
print(f'  raw_data:  {len(tensor_proto.raw_data)} bytes')

# Serialize to bytes
tensor_bytes = tensor_proto.SerializeToString()
print(f'\nSerialized to {len(tensor_bytes)} bytes')

# Save to disk
with open('weights.pb', 'wb') as f:
    f.write(tensor_bytes)
print(f'Saved to weights.pb ({os.path.getsize("weights.pb")} bytes)')

In [ ]:
# Load back: bytes → TensorProto → NumPy
loaded_proto = TensorProto()
with open('weights.pb', 'rb') as f:
    loaded_proto.ParseFromString(f.read())

loaded_array = to_array(loaded_proto)

print(f'Loaded name:  {loaded_proto.name}')
print(f'Loaded shape: {loaded_array.shape}')
print(f'Loaded dtype: {loaded_array.dtype}')
print(f'Values:\n{loaded_array}')
print(f'\nRound-trip lossless: {np.array_equal(original_array, loaded_array)}')

### Data Type Preservation

Protobuf serialization preserves the exact data type. Let's verify with different NumPy dtypes.

In [ ]:
test_dtypes = [
    ('float32', np.float32, TensorProto.FLOAT),
    ('float64', np.float64, TensorProto.DOUBLE),
    ('int32',   np.int32,   TensorProto.INT32),
    ('int64',   np.int64,   TensorProto.INT64),
    ('bool',    np.bool_,   TensorProto.BOOL),
]

print(f'{"dtype":>10s} | {"numpy_size":>10s} | {"proto_bytes":>11s} | {"roundtrip":>9s}')
print('-' * 50)

for name, np_dtype, onnx_type in test_dtypes:
    arr = np.array([1, 2, 3, 4, 5], dtype=np_dtype)
    proto = from_array(arr, name=f'test_{name}')
    serialized = proto.SerializeToString()

    restored = TensorProto()
    restored.ParseFromString(serialized)
    restored_arr = to_array(restored)

    ok = np.array_equal(arr, restored_arr) and arr.dtype == restored_arr.dtype
    print(f'{name:>10s} | {arr.nbytes:>10d} | {len(serialized):>11d} | {str(ok):>9s}')

<a id='section-5'></a>
## Section 5: Anatomy of a `.onnx` File

A `.onnx` file is just the raw bytes from `SerializeToString()`. There is no header, no magic number, no compression — it's pure protobuf.

Let's visualize the byte-level structure to understand what takes up space.

In [ ]:
# Build a model with embedded weights to see the size breakdown
W_data = np.random.randn(100, 50).astype(np.float32)  # 20,000 floats = 80 KB
b_data = np.random.randn(50).astype(np.float32)

W_init = numpy_helper.from_array(W_data, name='W')
b_init = numpy_helper.from_array(b_data, name='b')

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 100])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 50])

graph = make_graph(
    [make_node('MatMul', ['X', 'W'], ['XW']),
     make_node('Add', ['XW', 'b'], ['Y'])],
    'weighted', [X], [Y], [W_init, b_init])

model_with_weights = make_model(graph)
check_model(model_with_weights)

total_bytes = model_with_weights.SerializeToString()
weight_bytes = W_data.nbytes + b_data.nbytes
graph_bytes = len(total_bytes) - weight_bytes

print(f'Model file breakdown:')
print(f'  Weight data (W + b): {weight_bytes:>8,} bytes ({weight_bytes/len(total_bytes)*100:.1f}%)')
print(f'  Graph structure:     {graph_bytes:>8,} bytes ({graph_bytes/len(total_bytes)*100:.1f}%)')
print(f'  Total:               {len(total_bytes):>8,} bytes')

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
sizes = [weight_bytes, graph_bytes]
labels = [f'Weight Data\n{weight_bytes:,} bytes', f'Graph Structure\n{graph_bytes:,} bytes']
colors = ['#3498DB', '#E74C3C']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct='%1.1f%%',
    startangle=90, textprops={'fontsize': 11})
for t in autotexts:
    t.set_fontweight('bold')
ax.set_title('ONNX File Size Breakdown (100×50 weight matrix)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-6'></a>
## Section 6: All Serializable Proto Types

Every protobuf type in ONNX supports `SerializeToString()` / `ParseFromString()`. Here is the complete catalog:

| Proto Type | Description | Typical Standalone Use |
|-----------|-------------|------------------------|
| `ModelProto` | Complete model (graph + metadata) | Saving/loading `.onnx` files |
| `GraphProto` | Computation graph (nodes + I/O) | Subgraph extraction |
| `NodeProto` | Single operator node | Unit testing an operator |
| `TensorProto` | Tensor data + metadata | Saving weights, test data |
| `ValueInfoProto` | Type signature (name, dtype, shape) | Type checking |
| `AttributeProto` | Operator attribute (fixed param) | Rare |
| `FunctionProto` | Reusable function definition | Function libraries |
| `OpsetIdProto` | Opset domain + version | Rare |

Let's demonstrate serializing each independently.

In [ ]:
# Serialize individual components
components = {
    'ModelProto':     onnx_model,
    'GraphProto':     onnx_model.graph,
    'NodeProto':      onnx_model.graph.node[0],
    'ValueInfoProto': onnx_model.graph.input[0],
}

print(f'{"Proto Type":>18s} | {"Serialized Size":>15s} | {"Hex Preview (first 20 bytes)"}')
print('-' * 75)

for name, proto in components.items():
    data = proto.SerializeToString()
    preview = data[:20].hex()
    print(f'{name:>18s} | {len(data):>12d}  B | {preview}')

<a id='section-7'></a>
## Section 7: File Size Analysis

In real-world ONNX models, file size is dominated by **weight data**. The graph structure (nodes, edges, metadata) is negligible in comparison. Let's quantify this relationship.

In [ ]:
shapes = [(10, 5), (50, 25), (100, 50), (500, 250), (1000, 500), (2000, 1000)]
total_sizes = []
weight_sizes = []
overhead_sizes = []
param_counts = []

for rows, cols in shapes:
    W = numpy_helper.from_array(
        np.random.randn(rows, cols).astype(np.float32), name='W')
    b = numpy_helper.from_array(
        np.random.randn(cols).astype(np.float32), name='b')

    _X = make_tensor_value_info('X', TensorProto.FLOAT, [None, rows])
    _Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, cols])
    _g = make_graph(
        [make_node('MatMul', ['X', 'W'], ['XW']),
         make_node('Add', ['XW', 'b'], ['Y'])],
        'g', [_X], [_Y], [W, b])
    _m = make_model(_g)

    total = len(_m.SerializeToString())
    params = rows * cols + cols
    weight = params * 4  # float32 = 4 bytes
    overhead = total - weight

    total_sizes.append(total)
    weight_sizes.append(weight)
    overhead_sizes.append(overhead)
    param_counts.append(params)

    print(f'  W[{rows:>4d}×{cols:>4d}]  '
          f'params={params:>10,}  '
          f'total={total:>12,} B  '
          f'weights={weight:>12,} B  '
          f'overhead={overhead:>6,} B ({overhead/total*100:.1f}%)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: File size vs parameters
ax1.plot(param_counts, [s/1024 for s in total_sizes], 'o-', color='#2E86C1',
         linewidth=2, markersize=7, label='Total file size')
ax1.plot(param_counts, [s/1024 for s in weight_sizes], 's--', color='#E74C3C',
         linewidth=2, markersize=7, label='Weight data only')
ax1.set_xlabel('Number of Parameters', fontsize=12)
ax1.set_ylabel('File Size (KB)', fontsize=12)
ax1.set_title('File Size Scales Linearly with Parameters', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')
ax1.set_yscale('log')

# Right: Overhead percentage
overhead_pct = [o/t*100 for o, t in zip(overhead_sizes, total_sizes)]
ax2.bar(range(len(shapes)), overhead_pct, color='#E74C3C', alpha=0.7)
ax2.set_xlabel('Model Configuration', fontsize=12)
ax2.set_ylabel('Non-Weight Overhead (%)', fontsize=12)
ax2.set_title('Protobuf Overhead Shrinks with Model Size', fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(shapes)))
ax2.set_xticklabels([f'{r}×{c}' for r, c in shapes], rotation=45)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: The 2 GB Protobuf Limit

### The Problem

Protobuf has a hard limit: a single message cannot exceed **2 GB** ($2^{31}$ bytes). For ONNX, this means a `ModelProto` with all weights embedded cannot be larger than 2 GB.

### When Does This Matter?

A model with $N$ parameters stored as float32 (4 bytes each) hits the 2 GB limit at:

$$N_{\max} = \frac{2^{31}}{4} = 536{,}870{,}912 \approx 537\text{M parameters}$$

| Model | Parameters | Size (float32) | Over Limit? |
|-------|-----------|----------------|-------------|
| ResNet-50 | 25M | ~100 MB | No |
| BERT-large | 340M | ~1.3 GB | No |
| GPT-2 XL | 1.5B | ~6.0 GB | **Yes** |
| LLaMA-7B | 7B | ~28 GB | **Yes** |

### The Solution: External Data Format

ONNX supports storing weight data in **separate files** while the graph structure stays in the `.onnx` file. The model references external data by filename and byte offset.

```
Standard Format:                    External Data Format:
┌──────────────────┐               ┌──────────────────┐   ┌──────────────┐
│   model.onnx     │               │   model.onnx     │   │ model.onnx   │
│ ┌──────────────┐ │               │ ┌──────────────┐ │   │ _data        │
│ │  Graph       │ │               │ │  Graph       │ │   │              │
│ ├──────────────┤ │               │ │  (refs to    │ │   │ ┌──────────┐ │
│ │  Weights     │ │               │ │   external   │─┼──►│ │ Weights  │ │
│ │  (embedded)  │ │               │ │   data file) │ │   │ │ (raw     │ │
│ └──────────────┘ │               │ └──────────────┘ │   │ │  bytes)  │ │
└──────────────────┘               └──────────────────┘   │ └──────────┘ │
                                                          └──────────────┘
```

Using external data requires the `onnx.external_data_helper` module:

```python
from onnx.external_data_helper import convert_model_to_external_data

convert_model_to_external_data(
    model,
    all_tensors_to_one_file=True,
    location='model_weights.bin',
    size_threshold=1024  # tensors > 1KB go to external file
)
onnx.save(model, 'model.onnx')
```

In [ ]:
# Demonstrate the math behind the 2GB limit
limit_bytes = 2**31
bytes_per_param = {'float32': 4, 'float16': 2, 'int8': 1, 'bfloat16': 2}

print('Maximum parameters before hitting 2 GB protobuf limit:')
print('-' * 55)
for dtype_name, bpp in bytes_per_param.items():
    max_params = limit_bytes / bpp
    print(f'  {dtype_name:>8s}: {max_params:>15,.0f} params  '
          f'({max_params/1e9:.2f}B)')

<a id='section-9'></a>
## Section 9: Key Takeaways

### Serialization Cheat Sheet

| Task | Code |
|------|------|
| Save model to disk | `onnx.save(model, 'model.onnx')` |
| Load model from disk | `model = onnx.load('model.onnx')` |
| Model to bytes (in-memory) | `data = model.SerializeToString()` |
| Bytes to model (in-memory) | `model = onnx.load_model_from_string(data)` |
| NumPy → TensorProto | `proto = numpy_helper.from_array(arr, name='W')` |
| TensorProto → NumPy | `arr = numpy_helper.to_array(proto)` |
| Save tensor to disk | `open('w.pb','wb').write(proto.SerializeToString())` |
| Large model (>2 GB) | Use `convert_model_to_external_data()` |

### Critical Points

1. **Protobuf is the foundation** — every ONNX object is a protobuf message with `SerializeToString()` and `ParseFromString()`.

2. **Round-trips are lossless** — serializing then deserializing produces byte-identical results.

3. **File size ≈ weight size** — for any non-trivial model, >95% of the file is weight data.

4. **2 GB limit exists** — models with >~537M float32 parameters need the external data format.

5. **All proto types are serializable** — not just `ModelProto`, but also individual nodes, tensors, and graphs.

---

**Next:** [Initializers and Attributes](../03_Initializers_and_Attributes/) — Embed weights and set operator parameters.

In [ ]:
# Cleanup
import glob
for f in glob.glob('*.onnx') + glob.glob('*.pb'):
    os.remove(f)
    print(f'Removed {f}')